In [ ]:
import os
import json
import torch
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset
from dotenv import load_dotenv
from transformers import EarlyStoppingCallback

load_dotenv()

os.environ["WANDB_DISABLED"] = "true"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [2]:
TRAIN_DATASET = os.getenv("train_json")
OUTPUT_DIR = os.getenv("output")



In [ ]:
class ConversationDataset(TorchDataset):
    def __init__(self, samples):
        self.samples = samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

print("[INFO] Memuat & memproses dataset JSONL...")
fixed_samples = []
skipped = 0

with open(TRAIN_DATASET, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        sample = json.loads(line)
        new_messages = []
        valid = True
        
        for message in sample["messages"]:
            role = message["role"]
            content = message["content"]
            
            if role == "assistant":
                text = str(content)
                new_messages.append({"role": "assistant", "content": text})
            
            elif role == "user":
                new_content = []
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "image":
                        img_path = item["image"]
                        # Cek apakah gambar 
                        if not os.path.exists(img_path):
                            print(f"[WARNING] Gambar hilang: {img_path}")
                            valid = False
                            break
                        new_content.append({"type": "image", "image": img_path})
                    else:
                        new_content.append(item)
                
                if not valid:
                    break
                new_messages.append({"role": "user", "content": new_content})
        
        if valid:
            fixed_samples.append({"messages": new_messages})
        else:
            skipped += 1

print(f"[INFO] Total data VALID yang siap di-training: {len(fixed_samples)}")
print(f"[INFO] Data dilewati (gambar hilang): {skipped}")

# Split Dataset (90% Train, 10% Eval)
import random
random.seed(3407)
random.shuffle(fixed_samples)

split_idx = int(len(fixed_samples) * 0.9)
train_dataset = ConversationDataset(fixed_samples[:split_idx])
eval_dataset  = ConversationDataset(fixed_samples[split_idx:])

print(f"[INFO] Train split: {len(train_dataset)} | Eval split: {len(eval_dataset)}")

In [ ]:
print("[INFO] Memuat Model Qwen3-VL-2B...")
model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen3.5-2B",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("[INFO] Memasang LoRA Adapter...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, 
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    random_state=3407,
)

In [5]:
trainer = Trainer(
    model=model,
    data_collator=UnslothVisionDataCollator(model, tokenizer,),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=1,
        warmup_steps=20,
        num_train_epochs=10, 
        learning_rate=1e-5,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=None,
        load_best_model_at_end=True, 
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="paged_adamw_8bit",
        weight_decay=0.05,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
    ),
)

print("[INFO] Memulai Fine-Tuning...")
trainer_stats = trainer.train()

print(f"\n[INFO] Training selesai dalam waktu: {trainer_stats.metrics['train_runtime']:.2f} detik")

print(f"\n[INFO] Menyimpan LoRA Adapter terbaik ke: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\n[SELESAI] Model siap digunakan!")

Unsloth: Model does not have a default image size - using 512
[INFO] Memulai Fine-Tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,783 | Num Epochs = 10 | Total steps = 2,230
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 92,442,624 of 2,305,684,288 (4.01% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.482454,0.476055
100,0.410312,0.414287
150,0.396163,0.393214
200,0.378126,0.386228
250,0.356501,0.379005
300,0.356105,0.370657
350,0.354013,0.364239
400,0.347304,0.359326
450,0.339970,0.352850
500,0.328213,0.351257


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



[INFO] Training selesai dalam waktu: 11588.43 detik

[INFO] Menyimpan LoRA Adapter terbaik ke: /home/khai/Qwen3-VL/qwen3_vl/3-fine-tuning/hasil-fine-tune4-2b(new-label8)

[SELESAI] Model siap digunakan!
